# Annotations API Response → Parsing → Knowledge Graph

This notebook demonstrates the complete flow of biomedical annotation data:

1. **Raw API Response** (JSON-LD from Europe PMC Annotations API)
2. **Parsing** (extract entities, context, semantic URIs)
3. **RDF Conversion** (W3C Open Annotation model)
4. **Knowledge Graph** (queryable triples)

Follow along to see how mentions of diseases, chemicals, and genes from scientific papers are automatically extracted, linked to ontologies, and integrated into a knowledge graph.

## Step 1: Import Libraries

In [1]:
import json
from pprint import pprint
from rdflib import Graph, Namespace

from pyeuropepmc.processing.annotation_parser import parse_annotations
from pyeuropepmc.processing.annotations_to_rdf import annotations_to_rdf

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## Step 2: Raw Annotation API Response

This is real data from the Europe PMC Annotations API. The API returns a JSON-LD array with:

- **source**: Database source ("MED" for MEDLINE)
- **extId**: External identifier (PMID)
- **pmcid**: PMC identifier
- **annotations**: Array of extracted concept mentions with semantic URIs

In [2]:
# Real response from Europe PMC Annotations API
# This shows annotations from a ME/CFS research paper (PMID: 41444612)

RAW_API_RESPONSE = [
    {
        "source": "MED",
        "extId": "41444612",
        "pmcid": "PMC12888368",
        "annotations": [
            {
                "prefix": "and modifiers of",
                "exact": "ME/CFS",
                "postfix": "previously obscured by",
                "tags": [
                    {
                        "name": "chronic fatigue unspecified",
                        "uri": "http://linkedlifedata.com/resource/umls-concept/C0015674"
                    }
                ],
                "id": "http://europepmc.org/article/MED/41444612#europepmc-50b2a3b",
                "type": "Diseases",
                "section": "Abstract (http://purl.org/dc/terms/abstract)",
                "provider": "Europe PMC"
            },
            {
                "prefix": "regulation of glycolysis,",
                "exact": "amino acid",
                "postfix": "acid and lipid",
                "tags": [
                    {
                        "name": "amino acid",
                        "uri": "http://purl.obolibrary.org/obo/CHEBI_33709"
                    }
                ],
                "id": "http://europepmc.org/article/MED/41444612#europepmc-7e102098",
                "type": "Chemicals",
                "section": "Abstract (http://purl.org/dc/terms/abstract)",
                "provider": "Europe PMC"
            },
            {
                "prefix": "that can influence",
                "exact": "gene expression",
                "postfix": "expression profiles.",
                "tags": [
                    {
                        "name": "gene expression",
                        "uri": "http://identifiers.org/GO:0010467"
                    }
                ],
                "id": "http://europepmc.org/article/MED/41444612#europepmc-961800d8",
                "type": "Gene Ontology",
                "section": "Methods (http://purl.org/orb/Methods)",
                "provider": "Europe PMC"
            }
        ]
    }
]

print("Raw API Response (3 annotations from paper PMID:41444612):")
print("=" * 80)
pprint(RAW_API_RESPONSE, depth=2)

Raw API Response (3 annotations from paper PMID:41444612):
[{'annotations': [...],
  'extId': '41444612',
  'pmcid': 'PMC12888368',
  'source': 'MED'}]


## Step 3: Parse the API Response

The `parse_annotations()` function normalizes the raw API response:
- Extracts entities with semantic URIs
- Preserves contextual information
- Validates the response format
- Returns structured Python dictionaries

In [3]:
# Parse the raw API response
parsed = parse_annotations(RAW_API_RESPONSE)

print("\nParsed Result Structure:")
print("=" * 80)
print(f"Valid response: {parsed['valid']}")
print(f"Entities extracted: {len(parsed['entities'])}")
print(f"Relationships: {len(parsed['relationships'])}")
print(f"Sentences: {len(parsed['sentences'])}")
print()

if parsed.get('metadata'):
    print("Metadata:")
    pprint(parsed['metadata'])


Parsed Result Structure:
Valid response: True
Entities extracted: 3
Relationships: 0
Sentences: 0

Metadata:
{'page': 1, 'page_size': 3, 'source': '', 'total_count': 3, 'version': ''}


## Step 4: Inspect Extracted Entities

Each entity now includes structured fields for knowledge graph integration:
- `exact`: Text span from paper
- `name`: Normalized concept name
- `id`: Semantic URI to ontology (UMLS, CHEBI, GO, etc.)
- `annotation_category`: Type
- `section`: Where in the paper
- `prefix` / `postfix`: Surrounding text

In [4]:
print("\nAll Extracted Entities:")
print("=" * 80)

for idx, entity in enumerate(parsed['entities'], 1):
    print(f"\n[Entity {idx}]")
    print(f"  Text mention: '{entity.get('exact')}'")
    print(f"  Concept name: {entity.get('name')}")
    print(f"  Category: {entity.get('annotation_category')}")
    print(f"  Semantic URI: {entity.get('id')}")
    print(f"  Section: {entity.get('section')}")
    prefix = entity.get('prefix', '')[:20]
    postfix = entity.get('postfix', '')[:20]
    print(f"  Context: ...{prefix} [{entity.get('exact')}] {postfix}...")


All Extracted Entities:

[Entity 1]
  Text mention: 'ME/CFS'
  Concept name: chronic fatigue unspecified
  Category: Diseases
  Semantic URI: http://linkedlifedata.com/resource/umls-concept/C0015674
  Section: Abstract (http://purl.org/dc/terms/abstract)
  Context: ...and modifiers of [ME/CFS] previously obscured ...

[Entity 2]
  Text mention: 'amino acid'
  Concept name: amino acid
  Category: Chemicals
  Semantic URI: http://purl.obolibrary.org/obo/CHEBI_33709
  Section: Abstract (http://purl.org/dc/terms/abstract)
  Context: ...regulation of glycol [amino acid] acid and lipid...

[Entity 3]
  Text mention: 'gene expression'
  Concept name: gene expression
  Category: Gene Ontology
  Semantic URI: http://identifiers.org/GO:0010467
  Section: Methods (http://purl.org/orb/Methods)
  Context: ...that can influence [gene expression] expression profiles....


## Step 5: Convert to RDF (Knowledge Graph Format)

Transform parsed annotations into RDF triples following the W3C Open Annotation Data Model.

This creates:
- **oa:Annotation** instances for each mention
- **oa:hasBody** links to semantic concepts
- **oa:hasTarget** links to source articles
- **nif:Phrase** for textual expressions

In [5]:
# Convert to RDF graph
rdf_graph = annotations_to_rdf(parsed)

print("\nRDF Graph Created:")
print("=" * 80)
print(f"Total triples: {len(rdf_graph)}")
print(f"Total namespaces: {len(list(rdf_graph.namespaces()))}")
print()

# Show namespace prefixes
print("Namespaces bound in graph:")
for prefix, namespace in rdf_graph.namespaces():
    if prefix:  # Skip default namespace
        print(f"  {prefix}: {str(namespace)[:60]}...")


RDF Graph Created:
Total triples: 63
Total namespaces: 10

Namespaces bound in graph:
  owl: http://www.w3.org/2002/07/owl#...
  rdf: http://www.w3.org/1999/02/22-rdf-syntax-ns#...
  rdfs: http://www.w3.org/2000/01/rdf-schema#...
  xsd: http://www.w3.org/2001/XMLSchema#...
  xml: http://www.w3.org/XML/1998/namespace...
  oa: http://www.w3.org/ns/oa#...
  nif: http://persistence.uni-leipzig.org/nlp2rdf/ontologies/nif-co...
  dcterms: http://purl.org/dc/terms/...
  prov: http://www.w3.org/ns/prov#...
  pyeuropepmc: https://w3id.org/pyeuropepmc/vocab#...


## Step 6: View RDF Triples

Each RDF triple is a (subject, predicate, object) statement forming the knowledge graph.

In [6]:
# Display all triples
print("\nRDF Triples (Sample):")
print("=" * 80)

triples = list(rdf_graph)
print(f"Total: {len(triples)} triples\n")

# Show first 10 triples
for idx, (subject, predicate, obj) in enumerate(triples[:10], 1):
    print(f"Triple {idx}:")
    print(f"  S: {str(subject)[:70]}")
    print(f"  P: {str(predicate)[:70]}")
    print(f"  O: {str(obj)[:70]}")
    print()


RDF Triples (Sample):
Total: 63 triples

Triple 1:
  S: http://europepmc.org/article/MED/41444612#europepmc-961800d8
  P: http://www.w3.org/1999/02/22-rdf-syntax-ns#type
  O: http://www.w3.org/ns/oa#SpecificResource

Triple 2:
  S: http://europepmc.org/article/MED/41444612#europepmc-7e102098
  P: http://www.w3.org/1999/02/22-rdf-syntax-ns#type
  O: http://www.w3.org/ns/oa#SpecificResource

Triple 3:
  S: http://europepmc.org/article/MED/41444612#europepmc-961800d8
  P: https://w3id.org/pyeuropepmc/vocab#articleId
  O: PMC12888368

Triple 4:
  S: http://europepmc.org/article/MED/41444612#europepmc-7e102098
  P: https://w3id.org/pyeuropepmc/vocab#annotationType
  O: entity

Triple 5:
  S: http://europepmc.org/article/MED/41444612#europepmc-961800d8
  P: https://w3id.org/pyeuropepmc/vocab#textPrefix
  O: that can influence

Triple 6:
  S: http://europepmc.org/article/MED/41444612#europepmc-7e102098
  P: http://purl.org/dc/terms/isPartOf
  O: Abstract (http://purl.org/dc/terms/abstract)



## Step 7: Serialize to Turtle Format

Turtle is a human-readable RDF serialization format.

In [7]:
# Serialize to Turtle format
turtle = rdf_graph.serialize(format="turtle")

print("\nRDF in Turtle Format:")
print("=" * 80)
# Show first part
lines = turtle.split('\n')
for line in lines[:40]:  # First 40 lines
    print(line)

if len(lines) > 40:
    print(f"\n... ({len(lines) - 40} more lines)")


RDF in Turtle Format:
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix nif: <http://persistence.uni-leipzig.org/nlp2rdf/ontologies/nif-core#> .
@prefix oa: <http://www.w3.org/ns/oa#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix pyeuropepmc: <https://w3id.org/pyeuropepmc/vocab#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<http://europepmc.org/article/MED/41444612#europepmc-50b2a3b> a oa:Annotation,
        oa:SpecificResource,
        pyeuropepmc:EntityAnnotation ;
    rdfs:label "chronic fatigue unspecified" ;
    nif:anchorOf "ME/CFS" ;
    dcterms:creator "Europe PMC" ;
    dcterms:isPartOf "Abstract (http://purl.org/dc/terms/abstract)" ;
    owl:sameAs <http://linkedlifedata.com/resource/umls-concept/C0015674> ;
    oa:hasBody <http://linkedlifedata.com/resource/umls-concept/C0015674> ;
    oa:hasTarget <http://europepmc.org/article/MED/41444612>,
     

## Step 8: Query the Knowledge Graph

SPARQL queries on the RDF graph.

In [8]:
# Query to find all subjects
query_results = list(rdf_graph.subjects())

print("\nUnique Subject URIs in Graph:")
print("=" * 80)
print(f"Total: {len(set(query_results))}\n")

for subject in set(query_results):
    print(f"  {str(subject)[:70]}")


Unique Subject URIs in Graph:
Total: 3

  http://europepmc.org/article/MED/41444612#europepmc-961800d8
  http://europepmc.org/article/MED/41444612#europepmc-50b2a3b
  http://europepmc.org/article/MED/41444612#europepmc-7e102098


## Step 9: Summary - Complete Flow

### The Annotation Pipeline:

```
Raw API Response (JSON-LD)
          ↓
parse_annotations() - Extract & structure entities
          ↓
Parsed Python Dicts - Semantic URIs, context, metadata
          ↓
annotations_to_rdf() - Convert to W3C Open Annotation Model
          ↓
RDF Knowledge Graph - SPARQL-queryable triples
```

In [9]:
# Final summary
print("\n" + "=" * 80)
print("SUMMARY: Raw API → Parsed Data → RDF Knowledge Graph")
print("=" * 80)
print()
print("📥 Input: 1 API response document")
print(f"   - 3 annotations (ME/CFS, amino acid, gene expression)")
print()
print("🔍 Parsed: Extracted entities")
print(f"   - {len(parsed['entities'])} entities with semantic URIs")
print(f"   - Linked to UMLS, CHEBI, and Gene Ontology")
print(f"   - Context preserved (prefix/postfix/section)")
print()
print("📊 Knowledge Graph: RDF triples")
print(f"   - {len(rdf_graph)} triples created")
print(f"   - W3C Open Annotation model")
print(f"   - SPARQL-queryable")
print()
print("✓ Result: Automated biomedical concept extraction from literature!")


SUMMARY: Raw API → Parsed Data → RDF Knowledge Graph

📥 Input: 1 API response document
   - 3 annotations (ME/CFS, amino acid, gene expression)

🔍 Parsed: Extracted entities
   - 3 entities with semantic URIs
   - Linked to UMLS, CHEBI, and Gene Ontology
   - Context preserved (prefix/postfix/section)

📊 Knowledge Graph: RDF triples
   - 63 triples created
   - W3C Open Annotation model
   - SPARQL-queryable

✓ Result: Automated biomedical concept extraction from literature!
